<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大语言模型》（Build a Large Language Model From Scratch）</a> 一书的配套代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 5 章补充：在无标注数据上预训练 Llama 3


- 本 notebook 将 [Llama 3 1B 从零实现](../07_gpt_to_llama/standalone-llama32_ch.ipynb) 模型接入第 5 章的（预训练部分）
- 目的是展示如何将 Llama 3 1B 作为 [第 5 章](../01_main-chapter-code/ch05_ch.ipynb) 中使用的 GPT-2 模型的即插即用替代

In [ ]:
# pip install tokenizers


In [ ]:
from importlib.metadata import version

pkgs = [
    "matplotlib", 
    "numpy", 
    "tiktoken", 
    "torch",
       ]
for p in pkgs:
    print(f"{p} 版本: {version(p)}")


&nbsp;
## 5.1 评估生成式文本模型


- 无代码


&nbsp;
### 5.1.1 使用 Llama 3 生成文本

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/llama32.webp" width="700px">

In [ ]:
######################
### Llama 3 代码
######################
import torch
import torch.nn as nn


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.silu(x_fc1) * x_fc2
        return self.fc3(x)


def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, freq_config=None, dtype=torch.float32):
    assert head_dim % 2 == 0, "嵌入维度必须为偶数"

    # 计算逆频率
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))

    # 频率调整
    if freq_config is not None:
        low_freq_wavelen = freq_config["original_context_length"] / freq_config["low_freq_factor"]
        high_freq_wavelen = freq_config["original_context_length"] / freq_config["high_freq_factor"]

        wavelen = 2 * torch.pi / inv_freq

        inv_freq_llama = torch.where(
            wavelen > low_freq_wavelen, inv_freq / freq_config["factor"], inv_freq
        )

        smooth_factor = (freq_config["original_context_length"] / wavelen - freq_config["low_freq_factor"]) / (
            freq_config["high_freq_factor"] - freq_config["low_freq_factor"]
        )

        smoothed_inv_freq = (
            (1 - smooth_factor) * (inv_freq / freq_config["factor"]) + smooth_factor * inv_freq
        )

        is_medium_freq = (wavelen <= low_freq_wavelen) & (wavelen >= high_freq_wavelen)
        inv_freq_llama = torch.where(is_medium_freq, smoothed_inv_freq, inv_freq_llama)
        inv_freq = inv_freq_llama

    # 生成位置索引
    positions = torch.arange(context_length, dtype=dtype)

    # 计算角度
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # 形状: (context_length, head_dim // 2)

    # 扩展角度以匹配 head_dim
    angles = torch.cat([angles, angles], dim=1)  # 形状: (context_length, head_dim)

    # 预计算正弦与余弦
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


def apply_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "头维度必须为偶数"

    # 将 x 拆分为前半与后半
    x1 = x[..., : head_dim // 2]  # 前半
    x2 = x[..., head_dim // 2 :]  # 后半

    # 调整 sin 与 cos 的形状
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # 形状: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # 应用旋转变换
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    # 应用 cos/sin 旋转后可以使用较低精度
    return x_rotated.to(dtype=x.dtype)


class GroupedQueryAttention(nn.Module):
    def __init__(
            self, d_in, d_out, num_heads,
            num_kv_groups,
            dtype=None
        ):
        super().__init__()
        assert d_out % num_heads == 0, "d_out 必须能被 num_heads 整除"
        assert num_heads % num_kv_groups == 0, "num_heads 必须能被 num_kv_groups 整除"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_key = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=False, dtype=dtype)
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        self.W_query = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.out_proj = nn.Linear(d_out, d_out, bias=False, dtype=dtype)

    def forward(self, x, mask, cos, sin):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)  # 形状: (b, num_tokens, d_out)
        keys = self.W_key(x)  # 形状: (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)  # 形状: (b, num_tokens, num_kv_groups * head_dim)

        # 重塑 queries、keys 与 values
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim)

        # 转置 keys、values 与 queries
        keys = keys.transpose(1, 2)  # 形状: (b, num_kv_groups, num_tokens, head_dim)
        values = values.transpose(1, 2)  # 形状: (b, num_kv_groups, num_tokens, head_dim)
        queries = queries.transpose(1, 2)  # Shape: (b, num_heads, num_tokens, head_dim)

        # 应用 RoPE
        keys = apply_rope(keys, cos, sin)
        queries = apply_rope(queries, cos, sin)

        # 扩展 keys 与 values 以匹配头数
        # Shape: (b, num_heads, num_tokens, head_dim)
        keys = keys.repeat_interleave(self.group_size, dim=1)  # Shape: (b, num_heads, num_tokens, head_dim)
        values = values.repeat_interleave(self.group_size, dim=1)  # Shape: (b, num_heads, num_tokens, head_dim)
        # 例如，沿 dim=1（query 组）执行 repeat_interleave 之前：
        #   [K1, K2]
        # repeat_interleave 之后（每个 query 组重复 group_size 次）：
        #   [K1, K1, K2, K2]
        # 若使用普通 repeat 而非 repeat_interleave，会得到：
        #   [K1, K2, K1, K2]

        # 计算缩放点积注意力（即自注意力），并使用因果掩码
        # Shape: (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)  # 每个头的点积

        # 计算注意力分数
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        assert keys.shape[-1] == self.head_dim

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # 合并各头，其中 self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # 可选投影

        return context_vec


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            num_kv_groups=cfg["n_kv_groups"],
            dtype=cfg["dtype"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = nn.RMSNorm(cfg["emb_dim"], eps=1e-5, dtype=cfg["dtype"])
        self.norm2 = nn.RMSNorm(cfg["emb_dim"], eps=1e-5, dtype=cfg["dtype"])

    def forward(self, x, mask, cos, sin):
        # 注意力块的快捷连接
        shortcut = x
        x = self.norm1(x)
        x = self.att(x, mask, cos, sin)  # Shape [batch_size, num_tokens, emb_size]
        x = x + shortcut  # 加回原始输入

        # 前馈块的快捷连接
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut  # 加回原始输入

        return x


class Llama3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # 主模型参数
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        self.trf_blocks = nn.ModuleList(  # 使用 ModuleList，因为 Sequential 只能接受一个输入，而我们需要 `x, mask, cos, sin`
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = nn.RMSNorm(cfg["emb_dim"], eps=1e-5, dtype=cfg["dtype"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        # 取消注释以下代码以绑定权重
        # self.out_head.weight = self.tok_emb.weight
        # torch.nn.init.normal_(self.out_head.weight, mean=0.0, std=0.02)

        # 可复用工具
        cos, sin = compute_rope_params(
            head_dim=cfg["emb_dim"] // cfg["n_heads"],
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"],
            freq_config=cfg["rope_freq"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.cfg = cfg


    def forward(self, in_idx):
        # 前向传播
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds

        num_tokens = x.shape[1]
        mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1)
        
        for block in self.trf_blocks:
            x = block(x, mask, self.cos, self.sin)
        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits


In [ ]:
#######################
### 初始化 Llama 3
#######################

# Llama 3.2 1B

LLAMA32_CONFIG = {
    "vocab_size": 128_256,           # 词表大小
    "context_length": 131_072,       # 训练模型时使用的上下文长度
    "emb_dim": 2048,                 # 嵌入维度
    "n_heads": 32,                   # 注意力头数
    "n_layers": 16,                  # 层数
    "hidden_dim": 8192,              # FeedForward 中间层维度
    "n_kv_groups": 8,                # 分组查询注意力的 Key-Value 组数
    "rope_base": 500_000.0,          # RoPE 中 "theta" 的基数
    "dtype": torch.bfloat16,         # 较低精度 dtype，用于降低内存占用
    "rope_freq": {                   # RoPE 频率缩放
        "factor": 32.0,
        "low_freq_factor": 1.0,
        "high_freq_factor": 4.0,
        "original_context_length": 8192,
    }
}

LLAMA32_CONFIG["train_context_length"] = 256  # 数据集较小，同时希望控制内存占用

torch.manual_seed(123)
model = Llama3Model(LLAMA32_CONFIG)
model.eval();


In [ ]:
#######################
### 设置分词器
#######################

import os
from pathlib import Path

import tiktoken
from tiktoken.load import load_tiktoken_bpe



class Tokenizer:
    """围绕 tiktoken 的轻量封装，用于跟踪 Llama-3 特殊 token ID。"""
    def __init__(self, model_path):
        if not os.path.isfile(model_path):
            raise FileNotFoundError(model_path)

        mergeable = load_tiktoken_bpe(model_path)

        # 硬编码自 Meta 的 tokenizer.json
        self.special = {
            "<|begin_of_text|>": 128000,
            "<|end_of_text|>": 128001,
            "<|start_header_id|>": 128006,
            "<|end_header_id|>": 128007,
            "<|eot_id|>": 128009,
        }
        self.special.update({f"<|reserved_{i}|>": 128002 + i
                             for i in range(256)
                             if 128002 + i not in self.special.values()})

        self.model = tiktoken.Encoding(
            name=Path(model_path).name,
            pat_str=r"(?i:'s|'t|'re|'ve|'m|'ll|'d)"
                    r"|[^\r\n\p{L}\p{N}]?\p{L}+"
                    r"|\p{N}{1,3}"
                    r"| ?[^\s\p{L}\p{N}]+[\r\n]*"
                    r"|\s*[\r\n]+"
                    r"|\s+(?!\S)"
                    r"|\s+",
            mergeable_ranks=mergeable,
            special_tokens=self.special,
        )

    def encode(self, text, bos=False, eos=False):
        ids = ([self.special["<|begin_of_text|>"]] if bos else []) \
              + self.model.encode(text)
        if eos:
            ids.append(self.special["<|end_of_text|>"])
        return ids

    def decode(self, ids):
        return self.model.decode(ids)


- 请注意，Meta AI 要求你在下载文件前接受 Llama 3.2 许可条款；为此，你需要创建 Hugging Face Hub 账户并访问 [meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B) 仓库以接受条款
- 接下来，你需要创建访问令牌；要生成具有 READ 权限的访问令牌，请点击右上角头像并选择 "Settings"


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/settings.webp?1" width="300px">

- 然后，创建并复制访问令牌，以便在下一个代码单元中粘贴

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/access-token.webp?1" width="600px">

In [ ]:
# 若首次运行本 notebook，请取消注释并运行以下代码

#from huggingface_hub import login
#login()


In [ ]:
from huggingface_hub import hf_hub_download

tokenizer_file_path = hf_hub_download(
    repo_id="meta-llama/Llama-3.2-1B-Instruct",
    filename="original/tokenizer.model",
    local_dir="Llama-3.2-1B-Instruct"
)


In [ ]:
tokenizer = Tokenizer(tokenizer_file_path)


In [ ]:
# 与第 4 章相同

def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx 是当前上下文中的 (B, T) 索引数组
    for _ in range(max_new_tokens):

        # 若当前上下文超过支持的上下文大小，则裁剪
        # 例如，若 LLM 仅支持 5 个 token，而上下文大小为 10
        # 则仅最后 5 个 token 用作上下文
        idx_cond = idx[:, -context_size:]

        # 获取预测
        with torch.no_grad():
            logits = model(idx_cond)

        # 仅关注最后一个时间步
        # (batch, n_token, vocab_size) 变为 (batch, vocab_size)
        logits = logits[:, -1, :]

        # 获取 logits 值最大的词表条目索引
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        # 将采样索引追加到运行序列
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # 添加 batch 维度
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # 移除 batch 维度
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=LLAMA32_CONFIG["train_context_length"]
)

print("输出文本:\n", token_ids_to_text(token_ids, tokenizer))


&nbsp;
### 5.1.2 计算文本生成损失：交叉熵与困惑度


- 与第 5 章类似


&nbsp;
### 5.1.3 计算训练集与验证集损失


In [ ]:
import requests

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()


- 快速检查文本是否加载正常：打印前 99 个与后 99 个字符


In [ ]:
# 前 99 个字符
print(text_data[:99])


In [ ]:
# 后 99 个字符
print(text_data[-99:])


In [ ]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("字符数:", total_characters)
print("Token 数:", total_tokens)


In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # 对整个文本进行分词
        token_ids = tokenizer.encode(txt)

        # 使用滑动窗口将文本切分为 max_length 长度的重叠序列
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# 注意：下方函数需要修改，因为我们之前在数据加载器中硬编码了
# GPT-2 分词器
def create_dataloader_v1(txt, tokenizer, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    # 初始化分词器
    # tokenizer = tiktoken.get_encoding("gpt2")
    tokenizer = tokenizer

    # 创建数据集
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 创建 DataLoader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


# 训练/验证比例
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    tokenizer=tokenizer,
    batch_size=2,
    max_length=LLAMA32_CONFIG["train_context_length"],
    stride=LLAMA32_CONFIG["train_context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    tokenizer=tokenizer,
    batch_size=2,
    max_length=LLAMA32_CONFIG["train_context_length"],
    stride=LLAMA32_CONFIG["train_context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)


- 可选检查：数据是否加载正确：


In [ ]:
print("训练 DataLoader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\n验证 DataLoader:")
for x, y in val_loader:
    print(x.shape, y.shape)


- 另一个可选检查：token 数量是否在预期范围内：


In [ ]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("训练 token 数:", train_tokens)
print("验证 token 数:", val_tokens)
print("全部 token 数:", train_tokens + val_tokens)


- 接下来，我们实现一个工具函数，用于计算给定 batch 的交叉熵损失
- 此外，我们还实现第二个工具函数，用于计算数据加载器中指定 batch 数量的损失

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # 若 num_batches 超过数据加载器中的 batch 数，则将其缩减以匹配
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches


- 若你的机器有支持 CUDA 的 GPU，LLM 会在 GPU 上训练而无需修改代码
- 通过 `device` 设置，我们确保数据与 LLM 模型加载到同一设备上

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # 使用 PyTorch 2.9 或更新版本以获得稳定的 mps 结果
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")


print(f"使用 {device} 设备。")


model.to(device) # 对于 nn.Module 类，无需 model = model.to(device) 赋值


torch.manual_seed(123) # 由于数据加载器中的 shuffle，为可复现性设置随机种子

with torch.no_grad(): # 尚未训练，为效率关闭梯度跟踪
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("训练损失:", train_loss)
print("验证损失:", val_loss)


&nbsp;
## 5.2 训练 LLM


In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # 初始化列表以跟踪损失与已见 token 数
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # 主训练循环
    for epoch in range(num_epochs):
        model.train()  # 将模型设为训练模式
        
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # 重置上一 batch 迭代的损失梯度
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # 计算损失梯度
            optimizer.step() # 使用损失梯度更新模型权重
            tokens_seen += input_batch.numel()
            global_step += 1

            # 可选评估步骤
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # 每个 epoch 后打印样本文本
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.cfg["context_length"]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # 紧凑打印格式
    model.train()


In [ ]:
# 注意：
# 取消注释以下代码以计算执行时间
# import time
# start_time = time.time()

torch.manual_seed(123)
model = Llama3Model(LLAMA32_CONFIG)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 40
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

# 注意：
# 取消注释以下代码以显示执行时间
# end_time = time.time()
# execution_time_minutes = (end_time - start_time) / 60
# print(f"Training completed in {execution_time_minutes:.2f} minutes.")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # 绘制训练与验证损失随 epoch 的变化
    ax1.plot(epochs_seen, train_losses, label="训练损失")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="验证损失")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("损失")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # x 轴仅显示整数刻度

    # 创建第二个 x 轴表示已见 token 数
    ax2 = ax1.twiny()  # 创建与 y 轴共享的第二个 x 轴
    ax2.plot(tokens_seen, train_losses, alpha=0)  # 不可见绘图，用于对齐刻度
    ax2.set_xlabel("已见 token 数")

    fig.tight_layout()  # 调整布局以留出空间
    plt.savefig("loss-plot.pdf")
    plt.show()

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)


&nbsp;
## 5.3 控制随机性的解码策略


In [ ]:
inference_device = torch.device("cpu")

model.to(inference_device)
model.eval()

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(inference_device),
    max_new_tokens=25,
    context_size=LLAMA32_CONFIG["train_context_length"]
)

print("输出文本:\n", token_ids_to_text(token_ids, tokenizer))


In [ ]:
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Hello", tokenizer).to(inference_device),
    max_new_tokens=25,
    context_size=LLAMA32_CONFIG["train_context_length"]
)

print("输出文本:\n", token_ids_to_text(token_ids, tokenizer))


&nbsp;
### 5.3.1 温度缩放


- 与第 5 章类似


&nbsp;
### 5.3.2 Top-k 采样


- 与第 5 章类似


&nbsp;
### 5.3.3 修改文本生成函数


- 与第 5 章类似


&nbsp;
## 5.4 在 PyTorch 中加载与保存模型权重


- 与第 5 章类似


&nbsp;
## 5.5 加载预训练权重


- 请参阅 [Qwen3 0.6B 从零实现](../11_qwen3/standalone-qwen3_ch.ipynb)


&nbsp;
## 总结与要点


- 已跳过
